In [2]:
!pip install opencv-python

In [3]:
import ast
import glob
import shutil
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from skimage import io

# =============================================================================
# CONFIGURATION: Update ROOT to point to your local project directory.
# The folder structure expected is:
#   ROOT/
#     HemorrhageLabels/   <- CSV files with hand-annotated hemorrhage polygons
#     renders/            <- Raw CT scan images organized by hemorrhage type
#     MasksData-re/       <- Output folder for generated hemorrhage masks
#     CData-re/           <- Output folder for consolidated CT images
#     Normal3000-re/      <- Output folder for normal (non-hemorrhage) images
# =============================================================================
ROOT        = Path(r'path/to/your/project')  # Update this to your local path
LABEL_DIR   = ROOT / 'HemorrhageLabels'
RENDERS_DIR = ROOT / 'renders'
OUT_DIR     = ROOT / 'MasksData-re'
CDATA_DIR   = ROOT / 'CData-re' / 'brain_window'
NORMAL_DIR  = ROOT / 'Normal3000-re'
N_NORMAL    = 3000  # Number of normal images to sample for class balance

# Mapping from internal lowercase keys to folder name conventions in the dataset
Hemorrhage_Type = {
    'epidural':         'Epidural',
    'intraparenchymal': 'Intraparenchymal',
    'multiple':         'Multiple',
    'subarachnoid':     'Subarachnoid',
    'subdural':         'Subdural',
}

# Create all necessary output directories if they don't already exist
for p in [CDATA_DIR,
          OUT_DIR / 'bi-masks', OUT_DIR / 'qu-masks',
          NORMAL_DIR / 'images', NORMAL_DIR / 'bi-masks', NORMAL_DIR / 'qu-masks']:
    p.mkdir(parents=True, exist_ok=True)

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def find_csv(keyword: str) -> Path:
    """Locate a CSV file in LABEL_DIR whose filename contains the given keyword."""
    hit = list(LABEL_DIR.glob(f'*{keyword}*.csv'))
    if not hit:
        raise FileNotFoundError(f'No CSV found: {keyword}')
    return hit[0]

def image_path(origin: str, htype: str) -> Path:
    """Construct the full path to a CT image given its filename and hemorrhage type."""
    return RENDERS_DIR / htype / htype / 'brain_window' / origin

def chose_label(row) -> str:
    """
    Select the best available segmentation label for a given image.
    Prefers 'Correct Label' if it exists and contains valid polygon data,
    otherwise falls back to 'Majority Label' from crowd annotations.
    """
    correct = row['Correct Label']
    if pd.isnull(correct):
        return row['Majority Label']
    try:
        return row['Majority Label'] if not any(ast.literal_eval(correct)) else correct
    except Exception:
        return row['Majority Label']

def parse_point(label_str: str) -> list:
    """
    Parse a label string containing polygon coordinates into a list of numpy arrays.
    Each array represents one polygon (one hemorrhage region) as (x, y) coordinate pairs.
    Handles both single-polygon and multi-polygon label formats.
    """
    data = ast.literal_eval(label_str)
    if not data:
        return []
    # Single polygon stored as a flat list of dicts
    if len(data) > 15 and isinstance(data[0], dict):
        return [np.array([[p['x'], p['y']] for p in data])]
    # Multiple polygons stored as a list of lists
    result = []
    for segment in data:
        if isinstance(segment, list):
            result.append(np.array([[p['x'], p['y']] for p in segment]))
    return result

def scale_point(points: list, size: int) -> list:
    """
    Scale normalized polygon coordinates (0-1 range) to pixel coordinates.
    Coordinates in the CSV are stored as fractions of image width/height,
    so we multiply by the image size to get actual pixel positions.
    """
    return [(size * p).astype('int32') for p in points]

# =============================================================================
# MASK GENERATORS
# Two types of masks are generated for each image:
#
# 1. Binary mask: white (255) where hemorrhage exists, black (0) elsewhere.
#    Simple presence/absence signal used for basic segmentation tasks.
#
# 2. Quad-level mask: four distinct intensity levels representing anatomical regions:
#    - Black (0):        background (outside the skull)
#    - Dark gray (85):   hemorrhage region
#    - Medium gray (170): brain tissue
#    - White (255):      skull
#    This richer mask gives the U-Net more informative supervision during training,
#    helping it distinguish hemorrhage from surrounding brain structures.
# =============================================================================

def binary_mask(img_path: Path, points: list, out_path: Path):
    """
    Generate and save a binary mask for a CT image.
    Draws and fills the hemorrhage polygon(s) in white on a black canvas.
    """
    img = io.imread(str(img_path))
    canvas = np.zeros(img.shape, dtype='int32')
    scale = scale_point(points, img.shape[0])
    if scale:
        cv2.polylines(canvas, scale, isClosed=True, color=(255, 255, 255), thickness=1)
        cv2.fillPoly(canvas, scale, color=(255, 255, 255))
    tf.keras.utils.save_img(str(out_path), canvas)

def quad_mask(img_path: Path, points: list, out_path: Path):
    """
    Generate and save a quad-level anatomical mask for a CT image.
    Pixel intensities in the original image are used to distinguish skull and brain tissue.
    The hemorrhage polygon is drawn on top in a separate intensity level.
    """
    img = io.imread(str(img_path))
    canvas = np.zeros(img.shape, dtype='int32')
    # Use pixel intensity norms to classify anatomical regions
    norms = np.linalg.norm(img.astype(float), axis=2)
    canvas[norms > 425]                  = [255, 255, 255]  # skull (brightest)
    canvas[(norms > 20) & (norms < 360)] = [170, 170, 170]  # brain tissue
    # Overlay hemorrhage polygon on top of the anatomical segmentation
    scaled = scale_point(points, img.shape[0])
    if scaled:
        cv2.polylines(canvas, scaled, isClosed=True, color=(85, 85, 85), thickness=5)
        cv2.fillPoly(canvas, scaled, color=(85, 85, 85))  # hemorrhage region
    tf.keras.utils.save_img(str(out_path), canvas)

# =============================================================================
# LABEL PROCESSING
# Load and consolidate annotation CSVs for all hemorrhage types.
# Each CSV contains crowd-sourced polygon annotations for one hemorrhage type.
# =============================================================================

frames = []
for htype, keyword in Hemorrhage_Type.items():
    df = pd.read_csv(find_csv(keyword))
    df['Hemorrhage Type'] = htype
    df = df[['Origin', 'Majority Label', 'Correct Label', 'Hemorrhage Type']]
    df = df[df['Correct Label'].notnull() | df['Majority Label'].notnull()]
    frames.append(df)
raw = pd.concat(frames, ignore_index=True)

# Filter to only rows where the image file exists and has a valid (non-empty) label
valid_mask = raw.apply(
    lambda r: image_path(r['Origin'], r['Hemorrhage Type']).exists()
              and bool(ast.literal_eval(r['Majority Label'])),
    axis=1
)
clean = raw[valid_mask].copy().reset_index(drop=True)
print(f'Valid rows: {len(clean)}')
clean['Segmentation Label'] = clean.apply(chose_label, axis=1)
clean = clean.drop_duplicates('Segmentation Label', keep='first')

def flatten(lbl: str) -> str:
    """
    Normalize single-polygon labels that are wrapped in an extra list layer,
    ensuring consistent format across all label entries.
    """
    val = ast.literal_eval(lbl)
    if len(val) == 1 and isinstance(val[0], list):
        return str([pt for pt in val[0]])
    return lbl
clean['Segmentation Label'] = clean['Segmentation Label'].apply(flatten)

# =============================================================================
# HANDLE IMAGES WITH MULTIPLE HEMORRHAGE TYPES
# Some CT scans contain more than one type of hemorrhage. These appear as
# duplicate rows (same image, different hemorrhage types). We merge their
# polygon annotations into a single combined label and tag them as 'mix'.
# =============================================================================
dupe = clean[clean.duplicated('Origin', keep=False)]
merged = []
for origin, grp in dupe.groupby('Origin'):
    combined = []
    for lbl in grp['Segmentation Label']:
        pts = ast.literal_eval(lbl)
        combined += pts if len(pts) < 5 else [pts]
    merged.append({'Origin': origin, 'Hemorrhage Type': 'mix',
                   'Segmentation Label': str(combined)})

df = pd.concat([
    clean.drop_duplicates('Origin', keep=False),
    pd.DataFrame(merged)
], ignore_index=True)
df = df.drop_duplicates('Segmentation Label', keep='last').reset_index(drop=True)
df['Segmentation Label'] = df['Segmentation Label'].str.replace(', []', '', regex=False)

# Save the consolidated segmentation labels to CSV
df.to_csv(ROOT / 'segmentation-labels-re.csv', index=False)

# Save a separate classification labels CSV (image filename + hemorrhage type only)
cls_df = df[['Origin', 'Hemorrhage Type']].drop_duplicates('Origin').reset_index(drop=True)
cls_df.to_csv(ROOT / 'classification-labels-re.csv', index=False)
print(cls_df['Hemorrhage Type'].value_counts())

# Copy the relevant CT images into the consolidated CData directory
for _, row in cls_df.iterrows():
    src = image_path(row['Origin'], row['Hemorrhage Type'])
    if src.exists():
        shutil.copy(src, CDATA_DIR / row['Origin'])

# =============================================================================
# GENERATE MASKS FOR HEMORRHAGE IMAGES
# For each labeled hemorrhage image, generate both the binary and quad-level masks.
# =============================================================================
bi_dir = OUT_DIR / 'bi-masks'
qu_dir = OUT_DIR / 'qu-masks'

skipped = []
for i, row in df.iterrows():
    if i % 100 == 0:
        print(f'  {i}/{len(df)}')
    src_img = CDATA_DIR / row['Origin']
    if not src_img.exists():
        skipped.append(row['Origin'])
        continue
    try:
        pts = parse_point(row['Segmentation Label'])
        binary_mask(src_img, pts, bi_dir / row['Origin'])
        quad_mask(src_img, pts, qu_dir / row['Origin'])
    except Exception as e:
        skipped.append(f"{row['Origin']} ({e})")
if skipped:
    print(f'Skipped {len(skipped)} images')

# =============================================================================
# GENERATE MASKS FOR NORMAL (NON-HEMORRHAGE) IMAGES
# To address class imbalance, 3,000 normal CT scans are randomly sampled.
# Their masks are all-black (no hemorrhage present), giving the U-Net
# balanced exposure to both hemorrhage-positive and hemorrhage-negative examples.
# =============================================================================
normal = RENDERS_DIR / 'normal' / 'normal' / 'brain_window'
all_normals = [f for f in normal.iterdir() if f.suffix.lower() == '.jpg']
chosen = np.random.choice(all_normals, min(N_NORMAL, len(all_normals)), replace=False)

img_out = NORMAL_DIR / 'images'
bi_out  = NORMAL_DIR / 'bi-masks'
qu_out  = NORMAL_DIR / 'qu-masks'

for fpath in chosen:
    dst = img_out / fpath.name
    shutil.copy(fpath, dst)
    # Empty polygon list produces all-black masks for normal images
    binary_mask(dst, [], bi_out / fpath.name)
    quad_mask(dst, [], qu_out / fpath.name)

# Save the list of selected normal images for reproducibility
(ROOT / 'normals-re.txt').write_text('\n'.join(f.name for f in chosen))
